In [ ]:
# This notebook applies the following basic Machine Learning models:
# Logistic Regression, SVM, KNN and Decision Trees
###

In [ ]:
pip install zarr

In [ ]:
# 0. Preparation
###
# Importing libraries
import zarr
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2

from sklearn.metrics import classification_report, f1_score
from sklearn import linear_model, preprocessing
from sklearn.model_selection import train_test_split
from sklearn import svm, neighbors

In [ ]:
# Mounting GoogleDrive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# Reading data file from GoogleDrive
# Import  zarr
zarr_data = zarr.open("/content/drive/My Drive/Data Science/Team Project X-Rays/Dataframes/data_2.0_no_NA.zarr", mode='r')
zarr_target = zarr.open("/content/drive/My Drive/Data Science/Team Project X-Rays/Dataframes/target_2.0.zarr", mode='r')


In [ ]:
# Convert to pandas DataFrame
df = pd.DataFrame(zarr_data)
df_target = pd.DataFrame(zarr_target)
display(df.head())
display(df_target.head())

# Define data name
df_name = "Baseline 2.0"

# Define random subsample for computation efficiency
#df = df.sample(500)
#df_target = df_target.sample(500)


In [ ]:
# Explore Data
###

# Check data type is DataFrame
print(type(df))

# Show dimensions
print(df.shape)

# Show labels
print(df_target.value_counts())

# Attention! Class 1 here is Normal and Class 0 here is Covid (Other way around)

# Check distributions after normalisation
df.describe()

# after min-max normalisation there still is some variation in the means and std deviations

<class 'pandas.core.frame.DataFrame'>
(21165, 33371)
1.0    10192
2.0     6012
0.0     3616
3.0     1345
Name: count, dtype: int64


,0,1,2,3,4,5,6,7,8,9,...,33361,33362,33363,33364,33365,33366,33367,33368,33369,33370
count,21165.000000,21165.000000,21165.000000,21165.000000,21165.000000,21165.000000,21165.000000,21165.000000,21165.000000,21165.000000,...,21165.000000,21165.000000,21165.000000,21165.000000,21165.000000,21165.000000,21165.000000,21165.000000,21165.000000,21165.000000
mean,0.454181,0.452225,0.451806,0.451227,0.449380,0.449464,0.434717,0.432747,0.433448,0.433154,...,0.461985,0.439439,0.437007,0.438199,0.444468,0.446351,0.448798,0.445848,0.449427,0.450902
std,0.084192,0.084459,0.084301,0.084006,0.082997,0.082863,0.082354,0.082181,0.082559,0.082511,...,0.108263,0.109557,0.109028,0.109539,0.111641,0.111791,0.111114,0.108532,0.107861,0.106903
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.454181,0.452225,0.451806,0.451227,0.449380,0.449464,0.434717,0.432747,0.433448,0.433154,...,0.461985,0.439439,0.437007,0.438199,0.444468,0.446351,0.448798,0.445848,0.449427,0.450902
50%,0.454181,0.452225,0.451806,0.451227,0.449380,0.449464,0.434717,0.432747,0.433448,0.433154,...,0.461985,0.439439,0.437007,0.438199,0.444468,0.446351,0.448798,0.445848,0.449427,0.450902
75%,0.454181,0.452225,0.451806,0.451227,0.449380,0.449464,0.434717,0.432747,0.433448,0.433154,...,0.461985,0.439439,0.437007,0.438199,0.444468,0.446351,0.448798,0.445848,0.449427,0.450902
max,1.000000,0.978723,0.958824,0.964706,0.970588,0.980676,0.976471,0.975904,0.963855,0.970588,...,1.000000,1.000000,0.991453,0.992366,0.995652,0.991453,1.000000,0.994118,0.994118,1.000000


In [ ]:
# Check missing values
print(df.info())

print("Missing vars in columns:\n", df.isna().sum())
print("Number of total missing vars:", df.isna().sum().sum())
print("Number of total missing vars (% of all obs):", (df.isna().sum().sum())/(df.shape[0]*df.shape[1]))

# No missing vars. We can continue the ML modelling.

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21165 entries, 0 to 21164
Columns: 33371 entries, 0 to 33370
dtypes: float32(33371)
memory usage: 2.6 GB
None
Missing vars in columns:
 0        0
1        0
2        0
3        0
4        0
        ..
33366    0
33367    0
33368    0
33369    0
33370    0
Length: 33371, dtype: int64
Number of total missing vars: 0
Number of total missing vars (% of all obs): 0.0


In [ ]:
# 1. Data preprocessing
###

# Create categorical variable from Case
#df_target = df_target.replace({"Normal": 0, "COVID": 1, "Lung_Opacity": 2, "Viral Pneumonia": 3})
#df.Case.astype(int)

# Check construction
#print(df.Case.value_counts())

# Split data into target and features
#target = df.Case

# Features data: Drop Names and target
#data = df.drop(["Name", "Case"], axis = 1)
#data.head()
#data.shape

# Split data into target and features
data = zarr_data
target = zarr_target

# Split data into training and Test sets, save random state
X_train, X_test, y_train, y_test = train_test_split(data, target, test_size = 0.2, random_state = 123)

In [ ]:
# 2. Model 1 - Logistic Regression
###
import time
start_time = time.time()

# Instantiate Logistic regression for classification
clf1_name = "Logistic Regression"
clf1 = linear_model.LogisticRegression(solver='lbfgs', C = 1.0, max_iter = 10000)

# Train the model on training data
clf1.fit(X_train, y_train)

# Make predictions on test set
y_pred = clf1.predict(X_test)

# Calc accuracys
clf1_score = clf1.score(X_test, y_test)

# Calc mean unweighted F1 Score in all classes
clf1_f1 = f1_score(y_test, y_pred, average = "macro")

# Measure time
model1_time = (time.time() - start_time)/60

In [ ]:
# Show Results
###

# Modelling Time
print("Model 1: --- %s minutes ---" % model1_time)

# Score and F1-Score
print("The score is:", clf1_score)
print("The mean F1-Score (unweighted) is:", clf1_f1)

# Show Confusion Matrix
#cm1 = pd.crosstab(y_test, y_pred, rownames = ['Realised Class'], colnames = ['Predicted Class'])
#display(cm1)

# Show classification report
model1_cr = classification_report(y_test, y_pred)
print(model1_cr)


In [ ]:
# 3. Model 2 - linear SVM
###
import time
start_time = time.time()

# Instantiate SVM
clf2_name = "linear SVM"
clf2 = svm.SVC(gamma = 0.01, kernel = "poly")

# Train the model on training data
clf2.fit(X_train, y_train)

# Make predictions on test set
y_pred = clf2.predict(X_test)

# Calc accuracy
clf2_score = clf2.score(X_test, y_test)

# Calc mean unweighted F1 Score in all classes
clf2_f1 = f1_score(y_test, y_pred, average = "macro")

# Measure time
model2_time = (time.time() - start_time)/60

In [ ]:
# Show Results
###

# Modelling Time
print("Model 2: --- %s minutes ---" % model2_time)

# Score and F1-Score
print("The score is:", clf2_score)
print("The mean F1-Score (unweighted) is:", clf2_f1)

# Show Confusion Matrix
#cm2 = pd.crosstab(y_test, y_pred, rownames = ['Realised Class'], colnames = ['Predicted Class'])
#display(cm2)

# Show classification report
model2_cr = classification_report(y_test, y_pred)
print(model2_cr)

Model 2: --- 344.4692603826523 minutes ---
The score is: 0.7056461138672336
The mean F1-Score (unweighted) is: 0.6532765857453471
              precision    recall  f1-score   support

         0.0       0.41      0.43      0.42       694
         1.0       0.79      0.83      0.81      2027
         2.0       0.73      0.68      0.70      1236
         3.0       0.75      0.62      0.68       276

    accuracy                           0.71      4233
   macro avg       0.67      0.64      0.65      4233
weighted avg       0.71      0.71      0.71      4233



In [ ]:
# 3. Model 3 - KNN
###
from sklearn import neighbors
import time
start_time = time.time()

# Instantiate classifier
clf3_name = "KNN"
clf3 = neighbors.KNeighborsClassifier(n_neighbors = 7, metric = 'minkowski')

# Train the model on training data
clf3.fit(X_train, y_train)

# Make predictions on test set
y_pred = clf3.predict(X_test)

# Calc accuracys
clf3_score = clf3.score(X_test, y_test)

# Calc mean unweighted F1 Score in all classes
clf3_f1 = f1_score(y_test, y_pred, average = "macro")

# Measure time
model3_time = (time.time() - start_time)/60

In [ ]:
# Show Results
###

# Modelling Time
print("Model 3: --- %s minutes ---" % model3_time)

# Score and F1-Score
print("The score is:", clf3_score)
print("The mean F1-Score (unweighted) is:", clf3_f1)

# Show Confusion Matrix
#cm3 = pd.crosstab(y_test, y_pred, rownames = ['Realised Class'], colnames = ['Predicted Class'])
#display(cm3)

# Show classification report
model3_cr = classification_report(y_test, y_pred)
print(model3_cr)

Model 3: --- 7.292197199662526 minutes ---
The score is: 0.7011575714623198
The mean F1-Score (unweighted) is: 0.5962621814855813
              precision    recall  f1-score   support

         0.0       0.43      0.20      0.27       694
         1.0       0.74      0.88      0.80      2027
         2.0       0.68      0.74      0.71      1236
         3.0       0.92      0.44      0.60       276

    accuracy                           0.70      4233
   macro avg       0.69      0.57      0.60      4233
weighted avg       0.68      0.70      0.68      4233



In [ ]:
# 3. Model 4 - Decision Tree
###
from sklearn.tree import DecisionTreeClassifier
import time
start_time = time.time()

# Instantiate classifier
clf4_name = "Decision Tree"
clf4 = DecisionTreeClassifier(criterion = "entropy", max_depth = 4, random_state = 123)

# Train the model on training data
clf4.fit(X_train, y_train)

# Make predictions on test set
y_pred = clf4.predict(X_test)

# Calc accuracys
clf4_score = clf4.score(X_test, y_test)

# Calc mean unweighted F1 Score in all classes
clf4_f1 = f1_score(y_test, y_pred, average = "macro")

# Measure time
model4_time = (time.time() - start_time)/60


In [ ]:
# Show Results
###

# Modelling Time
print("Model 4: --- %s minutes ---" % model4_time)

# Score and F1-Score
print("The score is:", clf4_score)
print("The mean F1-Score (unweighted) is:", clf4_f1)

# Show Confusion Matrix
#cm4 = pd.crosstab(y_test, y_pred, rownames = ['Realised Class'], colnames = ['Predicted Class'])
#display(cm4)

# Show classification report
model4_cr = classification_report(y_test, y_pred)
print(model4_cr)
# Ideas: Could show most important Features here. But well, there are 4000 pixels...

Model 4: --- 7.965143013000488 minutes ---
The score is: 0.6113867233640444
The mean F1-Score (unweighted) is: 0.41571885498633343
              precision    recall  f1-score   support

         0.0       0.00      0.00      0.00       694
         1.0       0.61      0.92      0.73      2027
         2.0       0.63      0.53      0.58      1236
         3.0       0.60      0.25      0.36       276

    accuracy                           0.61      4233
   macro avg       0.46      0.43      0.42      4233
weighted avg       0.51      0.61      0.54      4233



/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
